# 数据清洗与航班衔接 / Data cleaning and flight connections

将 2026 年 4—5 月 BTS 航班记录整理为全组共享的航班表和同机衔接表。每行航班保留原始来源；同一飞机按计划时间排序，建立相邻航段之间的连接。

有效衔接采用以下口径：两端均完成且未备降、机场连续、时间检查通过、计划周转为 0—24 小时。隔夜连接另行标记。

This notebook prepares shared flight and same-aircraft connection tables from BTS records for April–May 2026. Each flight retains its source identifier. Consecutive legs are linked after sorting each aircraft by scheduled departure time.

An eligible connection has two completed, non-diverted flights, matching connection airports, consistent timestamps and a scheduled turnaround of 0–24 hours. Overnight connections are flagged separately.

## 1. 环境与路径 / Environment and paths

选择自己的 Python 内核，从 `output_A` 或其上级项目目录运行。原始 ZIP 放在 `output_A` 上一级。Pandas 可直接读取导出的 `csv.gz`。

Select your Python kernel and run from `output_A` or the project directory above it. Keep the original ZIP files one level above `output_A`. Pandas reads the exported `csv.gz` files directly.

In [1]:
from pathlib import Path
from zoneinfo import ZoneInfo
import hashlib
import importlib.metadata
import json
import platform
import zipfile

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

possible_roots = [Path.cwd(), Path.cwd() / 'output_A', Path.cwd() / 'airline_delay' / 'output_A']
OUT = next((p for p in possible_roots if (p / 'references' / 'airportsdata_snapshot.csv').exists()), None)
assert OUT is not None, '请从项目目录运行 / Run from output_A, airline_delay or IND5003.'
OUT = OUT.resolve()
RAW = OUT.parent
for folder in ['data', 'tables', 'figures']:
    (OUT / folder).mkdir(exist_ok=True)
INPUT_FILES = [RAW / '2026_04.zip', RAW / '2026_05.zip']
assert all(p.is_file() for p in INPUT_FILES), '原始 ZIP 位于上一级 / Place the two raw ZIP files above output_A.'
MAX_SCHEDULED_TURN_MIN = 24 * 60
CLOCK_TOLERANCE_MIN = 1
pd.set_option('display.max_columns', 12)
pd.set_option('display.max_rows', 15)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
versions = {'python': platform.python_version()}
for package in ['pandas', 'numpy', 'matplotlib', 'seaborn', 'nbformat', 'nbclient', 'tzdata']:
    try:
        versions[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        versions[package] = '未安装 / Not installed; using system time-zone data'
display(pd.Series(versions, name='运行版本 / Package version').to_frame())

,运行版本 / Package version
python,3.14.7
pandas,3.0.3
numpy,2.4.6
matplotlib,3.10.9
seaborn,0.13.2
nbformat,5.11.0
nbclient,0.11.0
tzdata,2026.3


## 2. 读取航班记录 / Load flight records

直接从 ZIP 读取所需字段。`flight_id` 由月份和原始 CSV 行号组成；重复业务键用标记列记录。

Read the required columns directly from each ZIP. The `flight_id` combines the source month and CSV row number. Duplicate business keys are recorded in a flag column.

In [2]:
TEXT_COLUMNS = ['FlightDate', 'Reporting_Airline', 'Tail_Number',
                'Flight_Number_Reporting_Airline', 'Origin', 'Dest', 'OriginState', 'DestState',
                'CRSDepTime', 'CRSArrTime', 'DepTime', 'ArrTime', 'CancellationCode']
NUMERIC_COLUMNS = ['DayOfWeek', 'OriginAirportID', 'DestAirportID', 'CRSElapsedTime',
                   'ActualElapsedTime', 'DepDelay', 'ArrDelay', 'DepDelayMinutes', 'ArrDelayMinutes',
                   'DepDel15', 'ArrDel15', 'Cancelled', 'Diverted', 'Distance',
                   'CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay']
USECOLS = TEXT_COLUMNS + NUMERIC_COLUMNS
frames, source_manifest = [], []
for path in INPUT_FILES:
    with zipfile.ZipFile(path) as archive:
        csv_names = [name for name in archive.namelist() if name.lower().endswith('.csv')]
        assert len(csv_names) == 1, f'{path.name} CSV 数量异常 / Unexpected CSV count'
        with archive.open(csv_names[0]) as stream:
            header = pd.read_csv(stream, nrows=0).columns.tolist()
        assert set(USECOLS).issubset(header), f'{path.name} 缺少必需字段 / Missing required columns'
        with archive.open(csv_names[0]) as stream:
            part = pd.read_csv(stream, usecols=USECOLS, dtype={c: 'string' for c in TEXT_COLUMNS},
                               low_memory=False)
        part['source_file'] = path.name
        part['source_row'] = np.arange(2, len(part) + 2)  # CSV 第 1 行为表头 / Row 1 is the CSV header
        part['flight_id'] = path.stem + ':' + part['source_row'].astype(str)
        frames.append(part)
        source_manifest.append({'file': path.name, 'member': csv_names[0], 'rows': len(part),
                                'sha256': hashlib.sha256(path.read_bytes()).hexdigest(),
                                'original_column_count': len(header)})
        (OUT / 'references' / f'{path.stem}_readme.html').write_bytes(archive.read('readme.html'))
flights = pd.concat(frames, ignore_index=True)
del frames, part
assert flights.flight_id.is_unique
assert len(flights) == sum(item['rows'] for item in source_manifest)
for column in TEXT_COLUMNS:
    flights[column] = flights[column].str.strip().replace('', pd.NA)
for column in NUMERIC_COLUMNS:
    flights[column] = pd.to_numeric(flights[column], errors='raise')
flights['FlightDate'] = pd.to_datetime(flights['FlightDate'], format='%Y-%m-%d', errors='coerce')
key = ['FlightDate', 'Reporting_Airline', 'Flight_Number_Reporting_Airline', 'Origin', 'Dest', 'CRSDepTime']
flights['duplicate_key'] = flights.duplicated(key, keep=False)
flights['normal_operation'] = flights.Cancelled.eq(0) & flights.Diverted.eq(0)
for column in ['Cancelled', 'Diverted', 'DepDel15', 'ArrDel15']:
    assert flights[column].dropna().isin([0, 1]).all(), f'{column} 非 0/1 值 / Values outside 0 or 1'
assert flights[['Cancelled', 'Diverted']].notna().all().all()
display(pd.DataFrame(source_manifest)[['file', 'rows', 'original_column_count']])
display(flights[['flight_id', 'FlightDate', 'Tail_Number', 'Origin', 'Dest', 'DepDelay', 'ArrDelay']].head())

,file,rows,original_column_count
0,2026_04.zip,597919,110
1,2026_05.zip,611735,110


,flight_id,FlightDate,Tail_Number,Origin,Dest,DepDelay,ArrDelay
0,2026_04:2,2026-04-17,N378DA,GSP,ATL,-9.00,-30.00
1,2026_04:3,2026-04-18,N3751B,GSP,ATL,-4.00,-8.00
2,2026_04:4,2026-04-19,N3738B,GSP,ATL,21.00,1.00
3,2026_04:5,2026-04-20,N3738B,GSP,ATL,-3.00,-20.00
4,2026_04:6,2026-04-21,N379DA,GSP,ATL,-6.00,-18.00


## 3. 机场时区与时间转换 / Airport time zones and timestamps

`0850` 表示当地 08:50，`2400` 表示次日 00:00。先将计划出发时间转为 UTC，再加计划飞行时长得到计划到达时间。实际时刻由计划时刻加有符号延误计算，提前对应负值。

机场时区来自包内固定快照；重复或不存在的夏令时当地时刻标为缺失。

`0850` means 08:50 local time; `2400` means midnight on the following day. Convert scheduled departure to UTC, then add scheduled elapsed time to obtain scheduled arrival. Actual timestamps equal scheduled timestamps plus signed delays, with negative values representing early operation.

Airport time zones use the bundled reference snapshot. Ambiguous or nonexistent local times at daylight-saving transitions are marked missing.

In [3]:
airport_source = pd.read_csv(OUT / 'references' / 'airportsdata_snapshot.csv', keep_default_na=False)
airport_codes = sorted(set(flights.Origin.dropna()) | set(flights.Dest.dropna()))
airports = airport_source.loc[airport_source.iata.isin(airport_codes),
                             ['iata', 'icao', 'name', 'country', 'lat', 'lon', 'tz']].copy()
assert not airports.iata.duplicated().any(), '机场代码重复 / Duplicate IATA codes in airport reference'
missing_airports = sorted(set(airport_codes) - set(airports.iata))
assert not missing_airports, f'机场时区缺失 / Missing airport time zones: {missing_airports}'
for tz in airports.tz.unique():
    ZoneInfo(tz)
airports.to_csv(OUT / 'references' / 'airport_timezones_used.csv', index=False, encoding='utf-8-sig')
tz_lookup = airports.set_index('iata')['tz']
flights['origin_tz'] = flights.Origin.map(tz_lookup)
flights['dest_tz'] = flights.Dest.map(tz_lookup)

def hhmm_minutes(values):
    '''当地 HHMM 转为午夜后分钟 / Convert local HHMM to minutes after midnight; 2400=1440; invalid values become NaN.'''
    numbers = pd.to_numeric(values, errors='coerce').astype(float)
    regular = numbers.between(0, 2359) & numbers.mod(1).eq(0) & numbers.mod(100).lt(60)
    valid = regular | numbers.eq(2400)
    return (numbers.floordiv(100) * 60 + numbers.mod(100)).where(valid)

def local_schedule_to_utc(dates, hhmm, timezones):
    local = pd.to_datetime(dates, errors='coerce').astype('datetime64[ns]') + pd.to_timedelta(hhmm_minutes(hhmm), unit='m')
    utc = pd.Series(pd.NaT, index=dates.index, dtype='datetime64[ns, UTC]')
    for tz in timezones.dropna().unique():
        ix = timezones.eq(tz).fillna(False)
        utc.loc[ix] = local.loc[ix].dt.tz_localize(str(tz), ambiguous='NaT', nonexistent='NaT').dt.tz_convert('UTC')
    return utc

def local_clock_error(utc, hhmm, timezones):
    '''比较当地钟表分钟的环形差 / Compare local clock minutes using circular differences at midnight.'''
    expected = hhmm_minutes(hhmm).mod(1440)
    observed = pd.Series(np.nan, index=utc.index)
    for tz in timezones.dropna().unique():
        ix = timezones.eq(tz).fillna(False)
        local = utc.loc[ix].dt.tz_convert(str(tz))
        observed.loc[ix] = local.dt.hour * 60 + local.dt.minute
    difference = (observed - expected).abs()
    return np.minimum(difference, 1440 - difference)

display(airports[['iata', 'name', 'tz']].head(8))
print(f'机场映射 / Airport mapping: {len(airports)} / {len(airport_codes)}')

,iata,name,tz
11658,ABE,Lehigh Valley International Airport,America/New_York
11659,ABI,Abilene Regional Airport,America/Chicago
11660,ABQ,Albuquerque International Sunport Airport,America/Denver
11661,ABR,Aberdeen Regional Airport,America/Chicago
11665,ACK,Nantucket Memorial Airport,America/New_York
11668,ACT,Waco Regional Airport,America/Chicago
11669,ACV,California Redwood Coast-Humboldt County Airport,America/Los_Angeles
11670,ACY,Atlantic City International Airport,America/New_York


机场映射 / Airport mapping: 348 / 348


### 时间转换示例 / Timestamp examples

下面的小案例检查 HHMM、跨日、夏令时、跨时区和提前出发的计算。

These small examples check HHMM parsing, midnight rollover, daylight-saving transitions, time zones and early departures.

In [4]:
parsed = hhmm_minutes(pd.Series(['0005', '0126', '2359', '2400', '1260', '-1', '930.5', None]))
assert parsed.iloc[:4].tolist() == [5, 86, 1439, 1440]
assert parsed.iloc[4:].isna().all()
test_dates = pd.Series(pd.to_datetime(['2026-04-01', '2026-04-01', '2026-03-08', '2026-11-01']))
test_utc = local_schedule_to_utc(test_dates, pd.Series(['2400', '1000', '0230', '0130']),
                               pd.Series(['America/New_York'] * 4))
assert test_utc.iloc[0] == pd.Timestamp('2026-04-02 04:00:00Z')
assert test_utc.iloc[1] == pd.Timestamp('2026-04-01 14:00:00Z')
assert test_utc.iloc[2:].isna().all()
assert test_utc.iloc[1] + pd.Timedelta(minutes=-10) == pd.Timestamp('2026-04-01 13:50:00Z')
assert local_clock_error(test_utc.iloc[[1]] + pd.Timedelta(hours=5), pd.Series(['1200'], index=[1]),
                         pd.Series(['America/Los_Angeles'], index=[1])).iloc[0] == 0
print('时间示例通过 / Timestamp examples passed: HHMM, 2400, DST, early departure and time zones.')

时间示例通过 / Timestamp examples passed: HHMM, 2400, DST, early departure and time zones.


## 4. 生成并核对时间 / Construct and check timestamps

将推算时刻与原始当地时钟及实际飞行时长核对，允许 1 分钟误差。取消和备降记录继续保留在航班表中。非负延误用于项目指标，有符号延误用于时间计算。

Compare reconstructed timestamps with the original local clocks and actual elapsed time, using a one-minute tolerance. Cancelled and diverted flights remain in the flight table. Nonnegative delays are used for project metrics; signed delays are used for timestamp calculations.

In [5]:
flights['scheduled_dep_utc'] = local_schedule_to_utc(flights.FlightDate, flights.CRSDepTime, flights.origin_tz)
flights['scheduled_arr_utc'] = flights.scheduled_dep_utc + pd.to_timedelta(flights.CRSElapsedTime, unit='m')
flights['actual_dep_utc'] = flights.scheduled_dep_utc + pd.to_timedelta(flights.DepDelay, unit='m')
flights['actual_arr_utc'] = flights.scheduled_arr_utc + pd.to_timedelta(flights.ArrDelay, unit='m')
flights['scheduled_dep_hour'] = hhmm_minutes(flights.CRSDepTime).mod(1440).floordiv(60).astype('Int64')
flights['scheduled_arr_clock_error'] = local_clock_error(flights.scheduled_arr_utc, flights.CRSArrTime, flights.dest_tz)
flights['actual_dep_clock_error'] = local_clock_error(flights.actual_dep_utc, flights.DepTime, flights.origin_tz)
flights['actual_arr_clock_error'] = local_clock_error(flights.actual_arr_utc, flights.ArrTime, flights.dest_tz)
calculated_elapsed = (flights.actual_arr_utc - flights.actual_dep_utc).dt.total_seconds() / 60
flights['actual_elapsed_error'] = (calculated_elapsed - flights.ActualElapsedTime).abs()
flights['delay_values_valid'] = (
    (flights.DepDelay.clip(lower=0) - flights.DepDelayMinutes).abs().le(CLOCK_TOLERANCE_MIN)
    & (flights.ArrDelay.clip(lower=0) - flights.ArrDelayMinutes).abs().le(CLOCK_TOLERANCE_MIN)
    & flights.DepDel15.eq(flights.DepDelay.ge(15).astype(int))
    & flights.ArrDel15.eq(flights.ArrDelay.ge(15).astype(int))
    & flights.DepDelayMinutes.ge(0) & flights.ArrDelayMinutes.ge(0)
)
flights['time_valid'] = (
    flights.scheduled_dep_utc.notna() & flights.CRSElapsedTime.gt(0)
    & flights.ActualElapsedTime.gt(0) & calculated_elapsed.gt(0)
    & flights.scheduled_arr_clock_error.le(CLOCK_TOLERANCE_MIN)
    & flights.actual_dep_clock_error.le(CLOCK_TOLERANCE_MIN)
    & flights.actual_arr_clock_error.le(CLOCK_TOLERANCE_MIN)
    & flights.actual_elapsed_error.le(CLOCK_TOLERANCE_MIN)
    & flights.delay_values_valid
)
assert flights.FlightDate.notna().all(), '日期解析失败 / Date parsing failed'
assert flights.DayOfWeek.eq(flights.FlightDate.dt.dayofweek + 1).all(), '星期与日期不一致 / Weekday does not match the date'
time_checks = pd.DataFrame({
    '检查 / Check': ['计划出发 UTC 缺失 / Missing scheduled departure UTC', '计划到达时钟差异 / Scheduled arrival clock mismatch', '实际出发时钟差异 / Actual departure clock missing or mismatched',
             '实际到达时钟差异 / Actual arrival clock missing or mismatched', '实际历时差异 / Actual elapsed time missing or mismatched', '完成航班时间或延误差异 / Completed flights with time or delay discrepancies'],
    '记录数 / Rows': [flights.scheduled_dep_utc.isna().sum(),
               (~flights.scheduled_arr_clock_error.le(CLOCK_TOLERANCE_MIN)).sum(),
               (flights.normal_operation & ~flights.actual_dep_clock_error.le(CLOCK_TOLERANCE_MIN)).sum(),
               (flights.normal_operation & ~flights.actual_arr_clock_error.le(CLOCK_TOLERANCE_MIN)).sum(),
               (flights.normal_operation & ~flights.actual_elapsed_error.le(CLOCK_TOLERANCE_MIN)).sum(),
               (flights.normal_operation & ~flights.time_valid).sum()]})
time_checks.to_csv(OUT / 'tables' / 'time_checks.csv', index=False, encoding='utf-8-sig')
display(time_checks)
anomaly_columns = ['flight_id', 'Tail_Number', 'Origin', 'Dest', 'FlightDate',
                   'CRSDepTime', 'CRSArrTime', 'DepTime', 'ArrTime', 'DepDelay', 'ArrDelay',
                   'CRSElapsedTime', 'ActualElapsedTime', 'scheduled_arr_clock_error',
                   'actual_dep_clock_error', 'actual_arr_clock_error', 'actual_elapsed_error']
flights.loc[flights.normal_operation & ~flights.time_valid, anomaly_columns].to_csv(
    OUT / 'tables' / 'time_anomalies_normal_flights.csv', index=False, encoding='utf-8-sig')

,检查 / Check,记录数 / Rows
0,计划出发 UTC 缺失 / Missing scheduled departure UTC,0
1,计划到达时钟差异 / Scheduled arrival clock mismatch,21
2,实际出发时钟差异 / Actual departure clock missing or m...,0
3,实际到达时钟差异 / Actual arrival clock missing or mis...,15
4,实际历时差异 / Actual elapsed time missing or mismat...,16
5,完成航班时间或延误差异 / Completed flights with time or d...,17


## 5. 构建相邻航班连接 / Link consecutive flights

先按尾号与计划时间排序，再生成相邻候选并筛选。取消、备降和异常航段在序列中保留，参与连接筛选。计划时间无法定位的尾号另行标记。

`first_exclusion_reason` 记录首个未通过条件；`check_*` 分别记录每项条件。计划周转是两班计划时刻的间隔，实际地面时间是两班实际时刻的间隔。

Sort by aircraft tail number and scheduled time, generate adjacent candidates, then apply the eligibility rules. Cancelled, diverted and anomalous legs remain in the sequence and participate in the screening. Aircraft with unplaceable scheduled times are flagged.

`first_exclusion_reason` records the first failed condition; `check_*` records each individual condition. Scheduled turnaround uses planned timestamps, while actual ground time uses actual timestamps.

In [6]:
def crosses_local_date(arrival_utc, departure_utc, timezones):
    result = pd.Series(False, index=arrival_utc.index)
    for tz in timezones.dropna().unique():
        ix = timezones.eq(tz).fillna(False)
        arrival_date = arrival_utc.loc[ix].dt.tz_convert(str(tz)).dt.date
        departure_date = departure_utc.loc[ix].dt.tz_convert(str(tz)).dt.date
        result.loc[ix] = (arrival_utc.loc[ix].notna() & departure_utc.loc[ix].notna()
                          & arrival_date.ne(departure_date))
    return result


def build_connections(frame, max_turn=1440):
    '''连接同尾号相邻计划航段 / Link adjacent scheduled legs of each aircraft.'''
    ordered = frame.loc[frame.Tail_Number.notna()].copy()
    ordered = ordered.sort_values(['Tail_Number', 'scheduled_dep_utc', 'flight_id'], kind='stable').reset_index(drop=True)
    ordered['tail_schedule_incomplete'] = ordered.scheduled_dep_utc.isna().groupby(ordered.Tail_Number).transform('any')
    ordered['schedule_tie'] = ordered.duplicated(['Tail_Number', 'scheduled_dep_utc'], keep=False)
    known = ordered.loc[ordered.actual_dep_utc.notna()]
    prev_max = known.groupby('Tail_Number').actual_dep_utc.cummax().groupby(known.Tail_Number).shift()
    reverse = known.iloc[::-1]
    next_min = reverse.groupby('Tail_Number').actual_dep_utc.cummin().groupby(reverse.Tail_Number).shift().reindex(known.index)
    conflict = known.actual_dep_utc.le(prev_max) | known.actual_dep_utc.ge(next_min)
    ordered['actual_order_conflict'] = False
    ordered.loc[known.index, 'actual_order_conflict'] = conflict
    next_rows = ordered.groupby('Tail_Number', sort=False).shift(-1)
    has_next = next_rows.flight_id.notna()
    up, down = ordered.loc[has_next], next_rows.loc[has_next]
    pairs = pd.DataFrame(index=up.index)
    pairs['connection_id'] = up.flight_id + '->' + down.flight_id
    for label, src in [('upstream', up), ('downstream', down)]:
        for original, suffix in [('flight_id', 'flight_id'), ('Reporting_Airline', 'airline'), ('FlightDate', 'flight_date'),
                                 ('scheduled_dep_utc', 'scheduled_dep_utc'), ('scheduled_arr_utc', 'scheduled_arr_utc'),
                                 ('actual_dep_utc', 'actual_dep_utc'), ('actual_arr_utc', 'actual_arr_utc')]:
            pairs[f'{label}_{suffix}'] = src[original]
    pairs['tail_number'] = up.Tail_Number
    pairs['airport'] = up.Dest
    pairs['upstream_origin'] = up.Origin
    pairs['downstream_origin'] = down.Origin
    pairs['downstream_dest'] = down.Dest
    pairs['upstream_arr_delay'] = up.ArrDelay
    pairs['upstream_arr_delay_minutes'] = up.ArrDelayMinutes
    pairs['downstream_dep_delay'] = down.DepDelay
    pairs['downstream_dep_delay_minutes'] = down.DepDelayMinutes
    pairs['downstream_sched_hour'] = down.scheduled_dep_hour
    pairs['downstream_day_of_week'] = down.DayOfWeek
    pairs['downstream_distance'] = down.Distance
    pairs['downstream_crs_elapsed_time'] = down.CRSElapsedTime
    pairs['scheduled_turnaround_minutes'] = (down.scheduled_dep_utc - up.scheduled_arr_utc).dt.total_seconds() / 60
    pairs['actual_ground_minutes'] = (down.actual_dep_utc - up.actual_arr_utc).dt.total_seconds() / 60
    pairs['delta_delay_minutes'] = down.DepDelayMinutes - up.ArrDelayMinutes
    # 用当地日期识别隔夜 / Compare local calendar dates to identify overnight connections.
    pairs['overnight_scheduled'] = crosses_local_date(up.scheduled_arr_utc, down.scheduled_dep_utc, up.dest_tz)
    checks = {
        'tail_schedule_complete': ~up.tail_schedule_incomplete,
        'unique_schedule': ~(up.schedule_tie | down.schedule_tie.fillna(True) | up.duplicate_key | down.duplicate_key.fillna(True)),
        'normal_both': up.normal_operation & down.normal_operation.fillna(False),
        'time_valid_both': up.time_valid & down.time_valid.fillna(False),
        'actual_order_consistent': ~(up.actual_order_conflict | down.actual_order_conflict.fillna(True)),
        'airport_continuity': up.Dest.eq(down.Origin),
        'scheduled_gap_nonnegative': pairs.scheduled_turnaround_minutes.ge(0),
        'scheduled_gap_in_scope': pairs.scheduled_turnaround_minutes.le(max_turn),
        'actual_ground_nonnegative': pairs.actual_ground_minutes.ge(0),
    }
    pairs['first_exclusion_reason'] = 'accepted'
    for label, passed in checks.items():
        passed = passed.fillna(False).astype(bool)
        pairs[f'check_{label}'] = passed
        pairs.loc[pairs.first_exclusion_reason.eq('accepted') & ~passed, 'first_exclusion_reason'] = label
    pairs['main_analysis_eligible'] = pairs.first_exclusion_reason.eq('accepted')
    flag_cols = ['flight_id', 'tail_schedule_incomplete', 'schedule_tie', 'actual_order_conflict']
    return pairs.reset_index(drop=True), ordered[flag_cols]

### 衔接示例 / Connection examples

检查普通衔接、机场连续性、0 和 24 小时边界、取消与备降、重复时刻、飞行顺序及隔夜标签。

Check ordinary connections, airport continuity, the 0-hour and 24-hour boundaries, cancellations, diversions, duplicate times, flight ordering and overnight flags.

In [7]:
def fixture():
    dep = pd.to_datetime(['2026-04-01 09:00Z', '2026-04-01 11:00Z', '2026-04-01 13:00Z'])
    return pd.DataFrame({
        'flight_id': ['a', 'b', 'c'], 'Tail_Number': ['TEST'] * 3,
        'Reporting_Airline': ['XX'] * 3, 'FlightDate': pd.to_datetime(['2026-04-01'] * 3),
        'Origin': ['AAA', 'BBB', 'CCC'], 'Dest': ['BBB', 'CCC', 'DDD'], 'dest_tz': ['UTC'] * 3,
        'scheduled_dep_utc': dep, 'scheduled_arr_utc': dep + pd.Timedelta(hours=1),
        'actual_dep_utc': dep, 'actual_arr_utc': dep + pd.Timedelta(hours=1),
        'CRSDepTime': ['0900', '1100', '1300'], 'scheduled_dep_hour': [9, 11, 13],
        'ArrDelay': [0.] * 3, 'ArrDelayMinutes': [0.] * 3,
        'DepDelay': [0.] * 3, 'DepDelayMinutes': [0.] * 3,
        'DayOfWeek': [3] * 3, 'Distance': [100.] * 3, 'CRSElapsedTime': [60.] * 3,
        'normal_operation': [True] * 3, 'time_valid': [True] * 3, 'duplicate_key': [False] * 3})

f = fixture()
p, _ = build_connections(f)
assert len(p) == 2 and p.main_analysis_eligible.all() and p.scheduled_turnaround_minutes.eq(60).all()
for status in ['cancelled', 'diverted']:
    f = fixture(); f.loc[1, 'normal_operation'] = False
    p, _ = build_connections(f)
    assert not p.main_analysis_eligible.any() and 'a->c' not in p.connection_id.tolist(), status
f = fixture(); f.loc[1, 'Origin'] = 'ZZZ'
assert not build_connections(f)[0].iloc[0].main_analysis_eligible
f = fixture(); f.loc[1, 'actual_dep_utc'] = f.loc[0, 'actual_arr_utc']
assert build_connections(f)[0].iloc[0].main_analysis_eligible
f.loc[1, 'actual_dep_utc'] -= pd.Timedelta(minutes=1)
assert not build_connections(f)[0].iloc[0].main_analysis_eligible
for minutes, expected in [(1440, True), (1441, False), (-1, False)]:
    f = fixture().iloc[:2].copy()
    f.loc[1, 'scheduled_dep_utc'] = f.loc[0, 'scheduled_arr_utc'] + pd.Timedelta(minutes=minutes)
    f.loc[1, 'actual_dep_utc'] = f.loc[1, 'scheduled_dep_utc']
    assert bool(build_connections(f)[0].iloc[0].main_analysis_eligible) == expected
f = fixture(); f.loc[1, 'scheduled_dep_utc'] = f.loc[0, 'scheduled_dep_utc']
assert not build_connections(f)[0].main_analysis_eligible.any()
f = fixture(); f.loc[0, 'actual_dep_utc'] += pd.Timedelta(hours=5)
assert not build_connections(f)[0].main_analysis_eligible.any()
f = fixture(); f.loc[1, 'scheduled_dep_utc'] = pd.NaT
assert not build_connections(f)[0].main_analysis_eligible.any()
f = fixture(); f['Tail_Number'] = pd.NA
assert build_connections(f)[0].empty
dst_arr = pd.Series(pd.to_datetime(['2026-03-08 04:30Z', '2026-11-01 04:30Z']))
dst_dep = pd.Series(pd.to_datetime(['2026-03-08 07:30Z', '2026-11-01 06:15Z']))
assert crosses_local_date(dst_arr, dst_dep, pd.Series(['America/New_York'] * 2)).tolist() == [True, False]
f = fixture(); f.loc[0, ['ArrDelay', 'ArrDelayMinutes']] = [-30, 0]
f.loc[1, ['DepDelay', 'DepDelayMinutes']] = [-5, 0]
assert build_connections(f)[0].iloc[0].delta_delay_minutes == 0  # 非负延误变化 / Change in nonnegative delay
f = fixture(); f.loc[1, 'actual_dep_utc'] = pd.NaT
f.loc[0, 'actual_dep_utc'] = f.loc[2, 'actual_dep_utc'] + pd.Timedelta(minutes=1)
assert build_connections(f)[1].actual_order_conflict.tolist() == [True, False, True]
print('衔接示例通过 / Connection examples passed: interruptions, bounds, ordering, missing values and DST.')

衔接示例通过 / Connection examples passed: interruptions, bounds, ordering, missing values and DST.


## 6. 生成候选表与有效表 / Create candidate and eligible tables

候选表保存相邻航段及筛选结果；有效表用于后续分析。4 月末到 5 月初的跨月衔接一并保留。

The candidate table stores adjacent legs and screening results. The eligible table is used for downstream analysis. Connections across the April–May boundary are retained.

In [8]:
candidates, sequence_flags = build_connections(flights, MAX_SCHEDULED_TURN_MIN)
flights = flights.merge(sequence_flags, on='flight_id', how='left', validate='one_to_one')
connections = candidates.loc[candidates.main_analysis_eligible].copy()
assert candidates.connection_id.is_unique and connections.connection_id.is_unique
assert connections.upstream_flight_id.is_unique and connections.downstream_flight_id.is_unique
assert connections.airport.eq(connections.downstream_origin).all()
assert connections.scheduled_turnaround_minutes.between(0, MAX_SCHEDULED_TURN_MIN).all()
assert connections.actual_ground_minutes.ge(0).all()
assert len(flights) == sum(item['rows'] for item in source_manifest)
expected_candidates = flights.Tail_Number.notna().sum() - flights.Tail_Number.nunique()
assert len(candidates) == expected_candidates
reason_counts = candidates.first_exclusion_reason.value_counts().rename_axis('reason').reset_index(name='pairs')
reason_counts['share_of_candidates'] = reason_counts.pairs / len(candidates)
assert reason_counts.pairs.sum() == len(candidates)
reason_counts.to_csv(OUT / 'tables' / 'connection_exclusions.csv', index=False, encoding='utf-8-sig')
all_failures = pd.DataFrame({'check': [c for c in candidates if c.startswith('check_')],
                             'failed_pairs': [(~candidates[c]).sum() for c in candidates if c.startswith('check_')]})
all_failures.to_csv(OUT / 'tables' / 'connection_check_failures.csv', index=False, encoding='utf-8-sig')
display(reason_counts)
display(Markdown(f'候选 / Candidates: **{len(candidates):,}**; 有效 / Eligible: **{len(connections):,}**; 保留比例 / Retained: **{len(connections)/len(candidates):.2%}**.'))

,reason,pairs,share_of_candidates
0,accepted,1131645,0.94
1,normal_both,22432,0.02
2,scheduled_gap_in_scope,13808,0.01
3,airport_continuity,13653,0.01
4,scheduled_gap_nonnegative,12507,0.01
5,actual_order_consistent,8132,0.01
6,unique_schedule,543,0.00
7,time_valid_both,30,0.00
8,actual_ground_nonnegative,4,0.00


候选 / Candidates: **1,202,754**; 有效 / Eligible: **1,131,645**; 保留比例 / Retained: **94.09%**.

## 7. 汇总与样本 / Summaries and samples

汇总航班量、衔接量、缺失值以及各月份和航司的入选比例。首个排除原因表中的类别互斥；逐项检查表允许同一候选出现在多个检查中。

Summarise flight counts, connections, missing values and selection coverage by month and airline. First-exclusion categories are mutually exclusive; individual check counts may include the same candidate more than once.

In [9]:
summary = {
    'raw_flight_rows': len(flights), 'duplicate_key_rows': int(flights.duplicate_key.sum()),
    'missing_tail_rows': int(flights.Tail_Number.isna().sum()),
    'cancelled_rows': int(flights.Cancelled.sum()), 'diverted_rows': int(flights.Diverted.sum()),
    'normal_operation_rows': int(flights.normal_operation.sum()),
    'normal_time_valid_rows': int((flights.normal_operation & flights.time_valid).sum()),
    'airport_count': len(airport_codes), 'airline_count': int(flights.Reporting_Airline.nunique()),
    'candidate_connections': len(candidates), 'accepted_connections': len(connections),
    'accepted_share_of_candidates': len(connections) / len(candidates),
    'accepted_overnight_connections': int(connections.overnight_scheduled.sum()),
    'accepted_delayed_upstream_connections': int(connections.upstream_arr_delay_minutes.ge(15).sum()),
    'accepted_cross_month_connections': int(connections.upstream_flight_date.dt.month.ne(connections.downstream_flight_date.dt.month).sum()),
}
pd.DataFrame(summary.items(), columns=['metric', 'value']).to_csv(OUT / 'tables' / 'quality_summary.csv', index=False)
(OUT / 'tables' / 'quality_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2))
missing = flights[USECOLS].isna().sum().rename_axis('field').reset_index(name='missing_count')
missing['missing_share'] = missing.missing_count / len(flights)
missing.to_csv(OUT / 'tables' / 'missing_values.csv', index=False, encoding='utf-8-sig')
flights['has_accepted_incoming_connection'] = flights.flight_id.isin(connections.downstream_flight_id)
coverage = flights.assign(month=flights.FlightDate.dt.strftime('%Y-%m')).groupby(['month', 'Reporting_Airline']).agg(
    flight_rows=('flight_id', 'size'), normal_rows=('normal_operation', 'sum'),
    accepted_downstream_rows=('has_accepted_incoming_connection', 'sum'))
coverage['coverage_of_all_flights'] = coverage.accepted_downstream_rows / coverage.flight_rows
coverage.to_csv(OUT / 'tables' / 'coverage_by_month_airline.csv', encoding='utf-8-sig')
samples = []
for reason, group in candidates.groupby('first_exclusion_reason', sort=True):
    sample = group.sample(min(5, len(group)), random_state=5003).copy()
    sample['sample_group'] = reason
    samples.append(sample)
for name, mask in [('overnight', connections.overnight_scheduled),
                   ('cross_month', connections.upstream_flight_date.dt.month.ne(connections.downstream_flight_date.dt.month)),
                   ('early_upstream_arrival', connections.upstream_arr_delay.lt(0)),
                   ('upstream_delay_180plus', connections.upstream_arr_delay_minutes.ge(180))]:
    sample = connections.loc[mask].head(5).copy(); sample['sample_group'] = name; samples.append(sample)
audit_samples = pd.concat(samples, ignore_index=True)
audit_samples.to_csv(OUT / 'tables' / 'connection_audit_samples.csv', index=False, encoding='utf-8-sig')
display(pd.DataFrame(summary.items(), columns=['metric', 'value']))
display(audit_samples[['sample_group', 'upstream_flight_id', 'downstream_flight_id', 'airport',
                       'scheduled_turnaround_minutes', 'actual_ground_minutes']].head(10))

,metric,value
0,raw_flight_rows,"1,209,654.00"
1,duplicate_key_rows,0.00
2,missing_tail_rows,942.00
3,cancelled_rows,"11,057.00"
4,diverted_rows,"3,045.00"
5,normal_operation_rows,"1,195,552.00"
6,normal_time_valid_rows,"1,195,535.00"
7,airport_count,348.00
8,airline_count,13.00
9,candidate_connections,"1,202,754.00"


,sample_group,upstream_flight_id,downstream_flight_id,airport,scheduled_turnaround_minutes,actual_ground_minutes
0,accepted,2026_05:285590,2026_05:286224,DFW,366.00,308.00
1,accepted,2026_04:592539,2026_04:391,IAH,56.00,70.00
2,accepted,2026_05:582888,2026_05:584799,LGA,77.00,81.00
3,accepted,2026_04:150728,2026_04:159252,ATL,31.00,29.00
4,accepted,2026_05:124693,2026_05:109873,ORD,82.00,64.00
5,actual_ground_nonnegative,2026_05:505592,2026_05:505028,MCO,40.00,-11.00
6,actual_ground_nonnegative,2026_05:433375,2026_05:433346,CHA,31.00,-5.00
7,actual_ground_nonnegative,2026_04:160588,2026_04:152815,DCA,58.00,-69.00
8,actual_ground_nonnegative,2026_04:55462,2026_04:55463,DAY,45.00,-37.00
9,actual_order_consistent,2026_05:453214,2026_05:455634,CID,368.00,-261.00


## 8. 保存共享数据 / Save shared datasets

输出三张压缩 CSV：全量航班、全部候选衔接、有效衔接。UTC 列使用 `pd.to_datetime(..., utc=True)` 重读，ID 和航班号按字符串读取。字段含义见双语数据字典。

Export three compressed CSV files: all flights, all candidate connections and eligible connections. Parse UTC columns with `pd.to_datetime(..., utc=True)` and read IDs and flight numbers as strings. Field meanings are listed in the bilingual data dictionary.

In [10]:
for name, frame in [('flights_clean', flights), ('connection_candidates', candidates), ('flight_connections', connections)]:
    frame.to_csv(OUT / 'data' / f'{name}.csv.gz', index=False,
                 compression={'method': 'gzip', 'compresslevel': 3, 'mtime': 0})
flight_sample = flights.sample(min(200, len(flights)), random_state=5003)
flight_sample.to_csv(OUT / 'tables' / 'flights_sample_200.csv', index=False, encoding='utf-8-sig')
manifest = {'input_files': source_manifest, 'environment': versions,
            'parameters': {'max_scheduled_turn_minutes': MAX_SCHEDULED_TURN_MIN,
                           'clock_tolerance_minutes': CLOCK_TOLERANCE_MIN},
            'reference': json.loads((OUT / 'references' / 'sources.json').read_text()),
            'summary': summary}
(OUT / 'run_manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2))
# 导出字段与类型 / Export field names and types; meanings are in data_dictionary.md.
schema = []
for name, frame in [('flights_clean', flights), ('connection_candidates', candidates), ('flight_connections', connections)]:
    for column in frame:
        schema.append({'table': name, 'field': column, 'dtype_in_notebook': str(frame[column].dtype)})
pd.DataFrame(schema).to_csv(OUT / 'tables' / 'schema.csv', index=False)
display(pd.DataFrame([{'file': p.name, 'size_MB': p.stat().st_size / 1e6} for p in (OUT / 'data').glob('*.csv.gz')]))

,file,size_MB
0,flight_connections.csv.gz,76.10
1,connection_candidates.csv.gz,80.97
2,flights_clean.csv.gz,75.66


## 9. 组员接续 / Team handoff

**B：**按衔接机场比较延误变化，并结合上游延误与计划周转解释差异。

**C、D：**在上游实际到达时预测下游出发延误，共用样本和时间划分。上游延误、计划周转及航班计划信息作为输入；下游延误作为目标，实际地面时间和延误变化用于事后分析。这里使用实际记录识别的同机关系。

下一步打开 `02_整体EDA.ipynb`。

**B:** Compare delay changes by connection airport, accounting for upstream delay and scheduled turnaround.

**C and D:** Predict downstream departure delay at the upstream flight’s actual arrival, using shared samples and chronological splits. Inputs include upstream delay, scheduled turnaround and flight schedule information. Downstream delay is the target; actual ground time and delay change support retrospective analysis. Aircraft links are identified from the recorded operations.

Continue with `02_整体EDA.ipynb`.